## Problem 1

In [ ]:
import numpy as np

In [ ]:
class Vasicek:

    def __init__(self,kappa,theta,sigma):
        self.kappa=kappa
        self.theta=theta
        self.sigma=sigma

In [ ]:
hw41dynamics = Vasicek(kappa=3,theta=0.05,sigma=0.03)

In [ ]:
class Bond:

    def __init__(self, T):
        self.T=T


In [ ]:
hw41contract = Bond(T=5)

In [ ]:
class FDexplicitEngine:

    def __init__(self, rMax, rMin, deltar, deltat, useUpwind):
        self.rMax=rMax
        self.rMin=rMin
        self.deltar=deltar
        self.deltat=deltat
        self.useUpwind=useUpwind

    def price_bond_vasicek(self,contract,dynamics):
    # You complete the coding of this function
    #
    # Returns array of all initial short rates,
    # and the corresponding array of zero-coupon
    # T-maturity bond prices

        T = contract.T
        N=round(T/self.deltat)
        if abs(N-T/self.deltat) > 1e-12:
            raise ValueError("Bad delta t")

        r=np.arange(self.rMax,self.rMin-self.deltar/2,-self.deltar)   #I'm making the FIRST indices of the array correspond to HIGH levels of r
        bondprice=np.ones(np.size(r))

        if self.useUpwind:
            qu=    #fill this in with an array.
            qd=    #fill this in with an array.
            qm=    #fill this in with an array.
        else:
            qu=    #fill this in with an array.
            qd=    #fill this in with an array.
            qm=    #fill this in with an array.

        for t in np.arange(N-1,-1,-1)*self.deltat:
            # Do not change any of the code in this loop

            bondprice[1:-1]=1/(1+r[1:-1]*self.deltat)*(qd[1:-1]*bondprice[2:]+qm[1:-1]*bondprice[1:-1]+qu[1:-1]*bondprice[:-2])
            # We are only calculating the interior grid points here, so
            # bondprice, r, qd, qm, and qu have [1:-1] indexes

            # For this contract, it is not obvious
            # what boundary conditions to use at the top and bottom,
            # so let us assume "linearity" boundary conditions
            bondprice[0]=2*bondprice[1]-bondprice[2]
            bondprice[-1]=2*bondprice[-2]-bondprice[-3]

        return (r, bondprice)

In [ ]:
hw41FD = FDexplicitEngine(rMax=0.35,rMin=-0.25,deltar=0.01,deltat=0.01,useUpwind=False)

In [ ]:
(r, bondprice) = hw41FD.price_bond_vasicek(hw41contract,hw41dynamics)

In [ ]:
np.set_printoptions(precision=4,suppress=True)
displayrows=(r<0.15+hw41FD.deltar/2) & (r>0.0-hw41FD.deltar/2)

In [ ]:
print(np.stack((r, bondprice),axis=1)[displayrows])

## Problem 2

In [1]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

In [2]:
class CEV:

    def __init__(self,volcoeff,alpha,rGrow,r,X0):
        self.volcoeff = volcoeff
        self.alpha = alpha
        self.rGrow = rGrow
        self.r = r
        self.X0 = X0


In [3]:
hw42dynamics = CEV(volcoeff=3, alpha=-0.5, rGrow=0, r=0.05, X0=100)

In [4]:
class Put:

    def __init__(self,T,K):
        self.T = T;
        self.K = K;

In [5]:
hw42contract=Put(T=0.25,K=100)

In [10]:
class FD_CrankNicolson_Engine:

    def __init__(self,XMax,XMin,deltaX,deltat):
        self.XMax=XMax
        self.XMin=XMin
        self.deltaX=deltaX
        self.deltat=deltat

    #You complete the coding of this function:
    def TicksAndMatricesCEV(self,T,dynamics):

        alpha, r, rGrow, volcoeff = dynamics.alpha, dynamics.r, dynamics.rGrow, dynamics.volcoeff

        N=round(T/self.deltat)
        if abs(N-T/self.deltat)>1e-12:
            raise ValueError('Bad time step')
        numX=round((self.XMax-self.XMin)/self.deltaX)+1
        if abs(numX-(self.XMax-self.XMin)/self.deltaX-1)>1e-12:
            raise ValueError('Bad time step')
        X=np.linspace(self.XMax,self.XMin,numX)    #The FIRST indices in this array are for HIGH levels of X
        tTicks = np.arange(N-1,-1,-1)*self.deltat

        ratio1 = self.deltat/self.deltaX
        ratio2 = self.deltat/self.deltaX**2

        f =    # You fill in with an array of the same size as X.
        g =    # You fill in with an array of the same size as X.
        h =    # You fill in with an array of the same size as X (or a scalar is acceptable here)

        F = 0.5*ratio2*f + 0.25*ratio1*g
        G =     ratio2*f - 0.50*self.deltat*h
        H = 0.5*ratio2*f - 0.25*ratio1*g

        #Right-hand-side matrix
        RHSmatrix = diags([H[:-1], 1-G, F[1:]], [1,0,-1], shape=(numX,numX), format="csr")

        #Left-hand-side matrix
        LHSmatrix = diags([-H[:-1], 1+G, -F[1:]], [1,0,-1], shape=(numX,numX), format="csr")
        # diags creates SPARSE matrices

        return(X, tTicks, LHSmatrix, RHSmatrix, H[-1], F[0])


    #You complete the coding of this function:
    def price_put_CEV(self,contract,dynamics):

        # returns array of all initial X levels,
        # and the corresponding array of put prices

        X, tTicks, LHSmatrix, RHSmatrix, bottomH, topF = self.TicksAndMatricesCEV(contract.T,dynamics)
        # The X array contains the _interior_ levels of the grid,
        # from the smallest XMin to the largest XMax
        # The boundary conditions are imposed one level _beyond_,
        # e.g. at X_lowboundary=XMin-deltaX, not at XMin.
        # To relate to lecture notation, X_lowboundary is X_{-J}
        # whereas XMin is X_{-J+1}

        putprice=np.maximum(contract.K-X,0)
        X_lowboundary=self.XMin-self.deltaX

        for t in tTicks:

            rhs = RHSmatrix @ putprice

            #Now let's add the boundary condition vectors.
            #They are nonzero only in the last component:
            rhs[-1]=rhs[-1]+2*bottomH*(contract.K-X_lowboundary)

            putprice =                   #You code this.

            # Three possibilities are:
            # 1. scipy.linalg.solve
            # 2. scipy.sparse.linalg.spsolve
            # 3. scipy.linalg.solve_banded
            #
            # 1. is not recommended here, because it does not take advantage of the fact that our matrix is mostly zeros.
            # 2. is more efficient (faster, and uses less storage) in this case, by recognizing our matrix's sparse structure.
            # 3. is the most efficient, by recognizing the banded (specifically, tridiagonal) matrix structure in our case.
            #
            # For convenience, I chose to set up RHSmatrix and LHSmatrix for approach 2 (spsolve)
            # because 3 (solve_banded) doesn't implement a banded matrix "class"
            # that supports matrix multiplication seamlessly (which approach 2 does)

            putprice = np.maximum(putprice, contract.K-X)

        return(X, putprice)

In [11]:
hw42FD = FD_CrankNicolson_Engine(XMax=200,XMin=50,deltaX=0.1,deltat=0.0005)

In [12]:
(X0_all, putprice) = hw42FD.price_put_CEV(hw42contract,hw42dynamics)

In [13]:
# pricer_put_CEV_CrankNicolson gives us option prices for ALL X0 from XMin to XMax
# But let's display only for a few X0 near 100:

displayStart = hw42dynamics.X0-hw42FD.deltaX*1.5
displayEnd   = hw42dynamics.X0+hw42FD.deltaX*1.5
displayrows  = (X0_all>displayStart) & (X0_all<displayEnd)
np.set_printoptions(precision=4, suppress=True)
print(np.stack((X0_all, putprice),axis=1)[displayrows])

[[100.1      8.5081]
 [100.       8.5565]
 [ 99.9      8.605 ]]
